In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import PowerTransformer

warnings.filterwarnings('ignore')
# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


In [4]:
# 读取数据
eye = pd.read_excel('E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\eye_热力图降落阶段数据.xlsx')
eye = eye.drop(columns=['Unnamed: 0', '性别'])
eye.受试者 = eye.受试者.replace({'王蓉蓉': '王榕榕'})
eye.受试者 = eye.受试者.replace({'祝娇娇': '朱娇娇'})
eye.受试者 = eye.受试者.replace({'毛华浩': '毛华灏'})
eye.受试者 = eye.受试者.replace({'董佳乐': '董嘉乐'})
eye[eye['组别'] == 'Control']['受试者'].unique()

array(['冯晓娅', '周鑫颜', '祖丽米热', '朱娇娇', '肖雨晨', '陈妍', '冯科嘉', '刘勇杰', '杜佳鑫',
       '毛华灏', '王子铭', '祝鹏程', '程腾宇', '胡浩男', '董嘉乐', '郭浚杰'], dtype=object)

In [5]:
def interpolate_to_full_days(df, value_columns,
                             group_cols=['组别', '受试者'],
                             day_col='飞行天数',
                             full_days=range(1, 8)):
    """
    线性拟合并补齐完整天数（1-7天）
    df : 输入数据
    value_columns : 需要插值的列（列表）
    group_cols : 分组列
    day_col : 表示天数的列
    """

    result = []

    for keys, subdf in df.groupby(group_cols):
        days = subdf[day_col].values

        # 对每个指标列分别拟合
        pred_values = {col: [] for col in value_columns}

        for col in value_columns:
            values = subdf[col].values

            # 若该人只有 1 天数据，则无法拟合 → 用这一点填满所有天
            if len(days) == 1:
                a, b = 0, values[0]
            else:
                a, b = np.polyfit(days, values, 1)

            pred_values[col] = a * np.array(list(full_days)) + b

        # 组装结果 dataframe
        for i, d in enumerate(full_days):
            row = {}

            # group info
            for g, v in zip(group_cols, keys if isinstance(keys, tuple) else (keys,)):
                row[g] = v

            row[day_col] = d

            # 插值后的指标
            for col in value_columns:
                row[col] = pred_values[col][i]

            result.append(row)

    return pd.DataFrame(result).sort_values(group_cols + [day_col])

In [6]:
eye_value_cols = ['AOI转换次数', '静态注释熵(SGE)', '眼跳注视熵(GTE)']

eye_full = interpolate_to_full_days(eye, value_columns=eye_value_cols)
eye_full.to_excel('E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\eye.xlsx', index=False)
eye = eye_full
eye

,组别,受试者,飞行天数,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE)
0,Alcohol,付瑞晗,1,5.964286,0.793663,0.297438
1,Alcohol,付瑞晗,2,10.500000,0.884336,0.313038
2,Alcohol,付瑞晗,3,15.035714,0.975010,0.328639
3,Alcohol,付瑞晗,4,19.571429,1.065683,0.344240
4,Alcohol,付瑞晗,5,24.107143,1.156356,0.359840
...,...,...,...,...,...,...
219,Control,陈妍,3,21.535714,1.521286,0.548103
220,Control,陈妍,4,18.071429,1.326411,0.441623
221,Control,陈妍,5,14.607143,1.131536,0.335144
222,Control,陈妍,6,11.142857,0.936661,0.228664


In [7]:
scl = pd.read_excel('E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\GSR_SCL绘制热力图的数据.xlsx')
scl = scl[scl['阶段'] == '降落']
scl = scl.drop(columns=['阶段'])
scl
# scl[scl['受试者'] == '肖雨晨']

,组别,受试者,飞行天数,SCL_mean_yj
576,Alcohol,付瑞晗,2,0.008527
577,Alcohol,付瑞晗,3,0.008875
578,Alcohol,付瑞晗,4,0.008457
579,Alcohol,付瑞晗,5,0.010428
580,Alcohol,付瑞晗,6,0.009591
...,...,...,...,...
762,Control,陈妍,3,0.004505
763,Control,陈妍,4,0.003004
764,Control,陈妍,5,-0.602839
765,Control,陈妍,6,0.004675


In [8]:
# expected_days = set(range(2, 8))
# 
# scl.groupby('受试者')['飞行天数'].apply(
#     lambda x: sorted(expected_days - set(x))
# ).reset_index(name='missing_days')


In [9]:
def interpolate_missing_days(df):
    """
    对每个受试者：
    - 用现有天数线性拟合
    - 外推补齐 day1
    - 插值补齐中间缺失天数（如 day3）
    - 最终返回完整的 1~7 天数据
    """
    result = []

    # grouping columns
    group_cols = ['组别', '受试者']

    for keys, subdf in df.groupby(group_cols):

        # 现有天数和数值
        days = subdf['飞行天数'].values
        values = subdf['SCL_mean_yj'].values

        # ---- 用现有天数做线性拟合 ----
        # 拟合 y = a * x + b
        a, b = np.polyfit(days, values, 1)

        # ---- 生成完整 1-7 天 ----
        full_days = np.arange(1, 8)

        # ---- 线性外推 + 插值 ----
        pred_values = a * full_days + b

        # 拼装 DataFrame
        for d, v in zip(full_days, pred_values):
            row = {
                '组别': keys[0],
                '受试者': keys[1],
                '飞行天数': d,
                'SCL_mean_yj': v
            }
            result.append(row)

    return pd.DataFrame(result).sort_values(['组别', '受试者', '飞行天数'])


df_full = interpolate_missing_days(scl)
df_full.to_excel('E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\scl.xlsx', index=False)
df_full
scl = df_full
scl

,组别,受试者,飞行天数,SCL_mean_yj
0,Alcohol,付瑞晗,1,0.008423
1,Alcohol,付瑞晗,2,0.008640
2,Alcohol,付瑞晗,3,0.008858
3,Alcohol,付瑞晗,4,0.009075
4,Alcohol,付瑞晗,5,0.009293
...,...,...,...,...
219,Control,陈妍,3,-0.070973
220,Control,陈妍,4,-0.088214
221,Control,陈妍,5,-0.105455
222,Control,陈妍,6,-0.122697


In [10]:
ppg = pd.read_excel('E:\pycharm all files\眼动数据处理\相关性分析\数据\原始数据\PPG分阶段格式对齐数据.xlsx')
ppg = ppg[ppg['event'] == '降落']
ppg = ppg.drop(columns=['event'])
ppg.columns = ['组别', '受试者', '飞行天数', 'HR', 'SDNN', 'RMSSD', 'AMP']
ppg.组别 = ppg.组别.replace({'A': 'Alcohol'})
ppg.组别 = ppg.组别.replace({'B': 'Control'})
ppg.reset_index(drop=True, inplace=True)
ppg.to_excel('E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\ppg.xlsx', index=False)
ppg

,组别,受试者,飞行天数,HR,SDNN,RMSSD,AMP
0,Alcohol,付瑞晗,1,17.819101,-3.260506,1.857342,-3100.994240
1,Alcohol,付瑞晗,2,5.119280,-15.197881,-21.082419,-2460.812152
2,Alcohol,付瑞晗,3,9.814463,11.623381,13.436624,-569.390018
3,Alcohol,付瑞晗,4,44.991362,25.830811,31.173141,-3871.876130
4,Alcohol,付瑞晗,5,10.378346,33.789581,41.231056,-985.276444
...,...,...,...,...,...,...,...
219,Control,陈妍,3,-2.053871,38.025352,38.134196,-1057.457602
220,Control,陈妍,4,-6.284982,-16.635726,-24.647037,-3145.236328
221,Control,陈妍,5,-1.159217,17.134339,17.823007,-967.018514
222,Control,陈妍,6,-5.534589,-22.075732,-34.098901,-6270.818380


In [11]:
# 从三张表任选一张提取受试者列表
people = scl[['组别', '受试者']].drop_duplicates()

# 完整天数
full_days = pd.DataFrame({'飞行天数': range(1, 8)})

# 224 行标准索引
full_index = people.merge(full_days, how='cross')


def check_missing(df, name):
    merged = full_index.merge(df, on=['组别', '受试者', '飞行天数'], how='left', indicator=True)

    missing = merged[merged['_merge'] == 'left_only'][['组别', '受试者', '飞行天数']]

    print(f"\n====================")
    print(f"【{name} 缺失检查】")
    print("====================")

    if missing.empty:
        print(f"✔ {name} 完整无缺失（224 行齐全）")
    else:
        print(f"❗ {name} 缺失以下受试者 × 天数：")
        print(missing)


check_missing(scl, 'SCL')
check_missing(eye, 'Eye')
check_missing(ppg, 'PPG')



【SCL 缺失检查】
✔ SCL 完整无缺失（224 行齐全）

【Eye 缺失检查】
✔ Eye 完整无缺失（224 行齐全）

【PPG 缺失检查】
✔ PPG 完整无缺失（224 行齐全）


In [12]:
eye[eye['受试者'] == '毛华灏']
# eye

,组别,受试者,飞行天数,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE)
154,Control,毛华灏,1,11.214286,2.785992,0.487794
155,Control,毛华灏,2,12.071429,2.348301,0.473164
156,Control,毛华灏,3,12.928571,1.910611,0.458535
157,Control,毛华灏,4,13.785714,1.472920,0.443905
158,Control,毛华灏,5,14.642857,1.035229,0.429276
159,Control,毛华灏,6,15.500000,0.597538,0.414646
160,Control,毛华灏,7,16.357143,0.159848,0.400017


In [13]:
# 先把 SCL + eye 合并
df_merge1 = pd.merge(
    scl,
    eye,
    on=['组别', '受试者', '飞行天数'],
    how='outer'  # 如果某些人某天有缺失，也不会丢
)

# 再与 PPG 合并
df_all = pd.merge(
    df_merge1,
    ppg,
    on=['组别', '受试者', '飞行天数'],
    how='outer'
)

# 排序一下
df_all = df_all.sort_values(['组别', '受试者', '飞行天数'])
df_all

,组别,受试者,飞行天数,SCL_mean_yj,AOI转换次数,静态注释熵(SGE),眼跳注视熵(GTE),HR,SDNN,RMSSD,AMP
0,Alcohol,付瑞晗,1,0.008423,5.964286,0.793663,0.297438,17.819101,-3.260506,1.857342,-3100.994240
1,Alcohol,付瑞晗,2,0.008640,10.500000,0.884336,0.313038,5.119280,-15.197881,-21.082419,-2460.812152
2,Alcohol,付瑞晗,3,0.008858,15.035714,0.975010,0.328639,9.814463,11.623381,13.436624,-569.390018
3,Alcohol,付瑞晗,4,0.009075,19.571429,1.065683,0.344240,44.991362,25.830811,31.173141,-3871.876130
4,Alcohol,付瑞晗,5,0.009293,24.107143,1.156356,0.359840,10.378346,33.789581,41.231056,-985.276444
...,...,...,...,...,...,...,...,...,...,...,...
219,Control,陈妍,3,-0.070973,21.535714,1.521286,0.548103,-2.053871,38.025352,38.134196,-1057.457602
220,Control,陈妍,4,-0.088214,18.071429,1.326411,0.441623,-6.284982,-16.635726,-24.647037,-3145.236328
221,Control,陈妍,5,-0.105455,14.607143,1.131536,0.335144,-1.159217,17.134339,17.823007,-967.018514
222,Control,陈妍,6,-0.122697,11.142857,0.936661,0.228664,-5.534589,-22.075732,-34.098901,-6270.818380


In [14]:
df_all.to_excel('E:\pycharm all files\眼动数据处理\相关性分析\数据\预处理后的数据\eye&scl&ppg.xlsx', index=False)